# Download CIFAR-100

In [1]:
!pip install -q huggingface_hub torchvision ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 52.1 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login

login()

In [3]:
import os
import torchvision
import torchvision.transforms as transforms

DATA_DIR = "/content/cifar-100"
os.makedirs(DATA_DIR, exist_ok=True)

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR100(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

print("Downloaded successfully!")
print("Train:", len(train_dataset))
print("Test :", len(test_dataset))

100%|██████████| 169M/169M [23:51<00:00, 118kB/s] 


Downloaded successfully!
Train: 50000
Test : 10000


In [4]:
import numpy as np

train_images = train_dataset.data
train_labels = np.array(train_dataset.targets)

test_images = test_dataset.data
test_labels = np.array(test_dataset.targets)

print("Train images:", train_images.shape)
print("Train labels:", train_labels.shape)
print("Test images :", test_images.shape)
print("Test labels :", test_labels.shape)
print("Dtype       :", train_images.dtype)

Train images: (50000, 32, 32, 3)
Train labels: (50000,)
Test images : (10000, 32, 32, 3)
Test labels : (10000,)
Dtype       : uint8


In [5]:
np.save(os.path.join(DATA_DIR, "train_images.npy"), train_images)
np.save(os.path.join(DATA_DIR, "train_labels.npy"), train_labels)
np.save(os.path.join(DATA_DIR, "test_images.npy"), test_images)
np.save(os.path.join(DATA_DIR, "test_labels.npy"), test_labels)

print("Saved files:")
print(os.listdir(DATA_DIR))

Saved files:
['train_labels.npy', 'cifar-100-python.tar.gz', 'test_labels.npy', 'cifar-100-python', 'test_images.npy', 'train_images.npy']


In [6]:
assert train_images.shape == (50000, 32, 32, 3)
assert train_labels.shape == (50000,)
assert test_images.shape == (10000, 32, 32, 3)
assert test_labels.shape == (10000,)

assert train_images.dtype == np.uint8
assert test_images.dtype == np.uint8

assert len(np.unique(train_labels)) == 100
assert len(np.unique(test_labels)) == 100

print(" CIFAR-100 verification passed")
print("Classes:", len(np.unique(train_labels)))
print("Training samples:", len(train_images))
print("Test samples:", len(test_images))

 CIFAR-100 verification passed
Classes: 100
Training samples: 50000
Test samples: 10000


In [7]:
from huggingface_hub import HfApi

REPO_ID = "Sakhiur/blue-bird-77"

api = HfApi()

api.upload_folder(
    folder_path=DATA_DIR,
    path_in_repo="cifar-100",
    repo_id=REPO_ID,
    repo_type="dataset",
)

print("Uploaded to Hugging Face!")
print(f"Folder: {REPO_ID}/cifar-100")

Uploaded to Hugging Face!
Folder: Sakhiur/blue-bird-77/cifar-100


# Load CIFAR-100 back from Hugging Face

In [8]:
from huggingface_hub import snapshot_download

REPO_ID = "Sakhiur/blue-bird-77"

local_dir = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns="cifar-100/*"
)

print(local_dir)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--Sakhiur--blue-bird-77/snapshots/8ccd592db421d2d1d455f2427e28ce5b10fb2c3e


In [9]:
import os
import numpy as np

CIFAR100_DIR = os.path.join(local_dir, "cifar-100")

train_images = np.load(
    os.path.join(CIFAR100_DIR, "train_images.npy")
)

train_labels = np.load(
    os.path.join(CIFAR100_DIR, "train_labels.npy")
)

test_images = np.load(
    os.path.join(CIFAR100_DIR, "test_images.npy")
)

test_labels = np.load(
    os.path.join(CIFAR100_DIR, "test_labels.npy")
)

print("Train images:", train_images.shape)
print("Train labels:", train_labels.shape)
print("Test images :", test_images.shape)
print("Test labels :", test_labels.shape)

Train images: (50000, 32, 32, 3)
Train labels: (50000,)
Test images : (10000, 32, 32, 3)
Test labels : (10000,)


In [10]:
class_names = [
    "apple", "aquarium_fish", "baby", "bear", "beaver",
    "bed", "bee", "beetle", "bicycle", "bottle",
    "bowl", "boy", "bridge", "bus", "butterfly",
    "camel", "can", "castle", "caterpillar", "cattle",
    "chair", "chimpanzee", "clock", "cloud", "cockroach",
    "couch", "crab", "crocodile", "cup", "dinosaur",
    "dolphin", "elephant", "flatfish", "forest", "fox",
    "girl", "hamster", "house", "kangaroo", "keyboard",
    "lamp", "lawn_mower", "leopard", "lion", "lizard",
    "lobster", "man", "maple_tree", "motorcycle", "mountain",
    "mouse", "mushroom", "oak_tree", "orange", "orchid",
    "otter", "palm_tree", "pear", "pickup_truck", "pine_tree",
    "plain", "plate", "poppy", "porcupine", "possum",
    "rabbit", "raccoon", "ray", "road", "rocket",
    "rose", "sea", "seal", "shark", "shrew",
    "skunk", "skyscraper", "snail", "snake", "spider",
    "squirrel", "streetcar", "sunflower", "sweet_pepper",
    "table", "tank", "telephone", "television", "tiger",
    "tractor", "train", "trout", "tulip", "turtle",
    "wardrobe", "whale", "willow_tree", "wolf", "woman",
    "worm"
]

assert len(class_names) == 100

print("Number of classes:", len(class_names))

Number of classes: 100


# Interactive Inspector

In [13]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

split_dropdown = widgets.Dropdown(
    options=["Train", "Test"],
    value="Train",
    description="Split:"
)

class_dropdown = widgets.Dropdown(
    options=[("All classes", -1)] +
            [(f"{i}: {name}", i) for i, name in enumerate(class_names)],
    value=-1,
    description="Class:"
)

count_slider = widgets.IntSlider(
    value=12,
    min=1,
    max=30,
    step=1,
    description="Images:"
)

seed_slider = widgets.IntSlider(
    value=42,
    min=0,
    max=9999,
    step=1,
    description="Seed:"
)

button = widgets.Button(
    description="Show Images",
    button_style="primary"
)

output = widgets.Output()


def show_images(_):
    with output:
        clear_output(wait=True)

        if split_dropdown.value == "Train":
            images = train_images
            labels = train_labels
        else:
            images = test_images
            labels = test_labels

        selected_class = class_dropdown.value

        if selected_class == -1:
            indices = np.arange(len(images))
        else:
            indices = np.where(labels == selected_class)[0]

        rng = np.random.default_rng(seed_slider.value)

        n = min(count_slider.value, len(indices))
        chosen = rng.choice(indices, size=n, replace=False)

        cols = 4
        rows = int(np.ceil(n / cols))

        plt.figure(figsize=(12, rows * 3))

        for i, idx in enumerate(chosen):
            plt.subplot(rows, cols, i + 1)

            plt.imshow(images[idx])
            plt.axis("off")

            label = labels[idx]
            plt.title(
                f"Index: {idx}\n{class_names[label]}",
                fontsize=10
            )

        plt.tight_layout()
        plt.show()


button.on_click(show_images)

display(
    widgets.VBox([
        split_dropdown,
        class_dropdown,
        count_slider,
        seed_slider,
        button,
        output
    ])
)

In [14]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output


split_dropdown = widgets.Dropdown(
    options=["Train", "Test"],
    value="Train",
    description="Split:"
)

index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(train_images) - 1,
    step=1,
    description="Index:"
)

previous_button = widgets.Button(
    description="← Previous"
)

next_button = widgets.Button(
    description="Next →"
)

random_button = widgets.Button(
    description="🎲 Random"
)

output = widgets.Output()


def update_slider(change=None):
    if split_dropdown.value == "Train":
        index_slider.max = len(train_images) - 1
    else:
        index_slider.max = len(test_images) - 1

    if index_slider.value > index_slider.max:
        index_slider.value = index_slider.max


def show_image(change=None):
    with output:
        clear_output(wait=True)

        if split_dropdown.value == "Train":
            images = train_images
            labels = train_labels
        else:
            images = test_images
            labels = test_labels

        idx = index_slider.value
        image = images[idx]
        label = labels[idx]

        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.axis("off")

        plt.title(
            f"{split_dropdown.value}\n"
            f"Index: {idx}\n"
            f"Label: {label} — {class_names[label]}",
            fontsize=14
        )

        plt.show()


def previous(_):
    if index_slider.value > 0:
        index_slider.value -= 1


def next_image(_):
    if index_slider.value < index_slider.max:
        index_slider.value += 1


def random_image(_):
    import random
    index_slider.value = random.randint(
        0,
        index_slider.max
    )


split_dropdown.observe(update_slider, names="value")
index_slider.observe(show_image, names="value")

previous_button.on_click(previous)
next_button.on_click(next_image)
random_button.on_click(random_image)

display(
    widgets.VBox([
        split_dropdown,
        index_slider,
        widgets.HBox([
            previous_button,
            next_button,
            random_button
        ]),
        output
    ])
)

show_image()